# Screen-tuned SmolVLM fork (stretch experiment)

LoRA fine-tune of SmolVLM-256M-Instruct on a 1:1 mix of ScreenQA (mobile,
human-annotated) and a SYNTHETIC DESKTOP set rendered by synth_desktop.py
(niri-style scenes, exact labels, fills the no-desktop-dataset gap).
Part of Pawline experiments/screen-fork - see RESEARCH.md for the honest
feasibility notes (the OCR hybrid beats stock SmolVLM on text questions;
this fork targets VISUAL questions where OCR is blind). NOT a shipped
default: the result must beat stock SmolVLM AND the OCR hybrid on the
fixed eval battery (eval_battery.py, docs/EXPERIMENTS.md 4d/4e/4f) before
it is used anywhere. How-to-run: experiments/screen-fork/RUN.md.

**Runtime**: Colab free tier: Runtime > Change runtime type > T4 GPU.
Kaggle: enable GPU (P100) in the notebook settings.

**Licenses**: SmolVLM Apache-2.0; ScreenQA annotations CC BY 4.0
(attribute Google); the Rico screenshots behind ScreenQA carry Rico's
own research terms (interactionmining.org/archive/rico/copyright.txt) -
read them before releasing anything trained on this. The synthetic
desktop set is CC BY 4.0 "Pawline synthetic desktop dataset".
If you add WebSight to the mix: CC-BY-4.0 + you must disclose its use
when releasing a model. Keep this notice in any released model card.


The generator also emits two purely visual families (a
top-bar status dot and a window progress bar): state questions are
answered by the compositor route now, so those are what a vision
model is still needed for.


In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes Pillow


In [ ]:
import torch, datasets
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - stop here')


## Data: ScreenQA train split

86K human-annotated QA over Rico mobile screenshots (CC BY 4.0). Mobile,
not desktop - the known domain gap from RESEARCH.md. We cap the training
subset so a free-tier run finishes in the session; raise N if you have
Kaggle's 30 h/week.


In [ ]:
N = 1500  # light config: ~3000 rows total with the synthetic set, so a
# free Colab session can actually finish; raise on Kaggle
ds = datasets.load_dataset('bevaya/RICO-ScreenQA', split='train').shuffle(seed=0).select(range(N))
ds


In [ ]:
def to_chat(ex):
    return {'messages': [
        {'role':'user','content':[{'type':'image'},{'type':'text','text':ex['question']}]},
        {'role':'assistant','content':[{'type':'text','text':ex['ground_truth'][0]}]},
    ], 'image': ex['image']}
ds = ds.map(to_chat, remove_columns=[c for c in ds.column_names if c not in ('messages','image')])


## Data: synthetic desktop set (fills the no-desktop-dataset gap)

ScreenQA is Android; Pawline answers questions about a niri DESKTOP.
`synth_desktop.py` (experiments/screen-fork) renders niri-style scenes
(tiled columns, waybar, focus border) from a scene graph, so every label
is exact - count, focus, color, clock, spatial, presence, and
text-reading questions included. Fetched from the Pawline repo branch
and generated here; mixed 1:1 with the ScreenQA subset. Train seeds
start at 0; the eval battery uses seeds >= 10000, so there is no
train/eval overlap.

The generator also emits two purely visual families (a
top-bar status dot and a window progress bar): state questions are
answered by the compositor route now, so those are what a vision
model is still needed for.


In [ ]:
# Colab has no DejaVu fonts by default and synth_desktop.py renders with them.
# Without this the generator dies on ImageFont.truetype('/usr/share/fonts/truetype/dejavu/...')
!apt-get -qq install -y fonts-dejavu-core > /dev/null
import os; print(sorted(os.listdir('/usr/share/fonts/truetype/dejavu'))[:4])

In [ ]:
import glob, json, os, subprocess, urllib.request
from PIL import Image

GEN_URL = 'https://raw.githubusercontent.com/linuxkiller9000/pawline/work/llm-vision-experiments/experiments/screen-fork/synth_desktop.py'
urllib.request.urlretrieve(GEN_URL, 'synth_desktop.py')
# Two resolutions: 1920x1080 and 1366x768 (the Dell N5010's native panel),
# so the fork is not resolution-specialized. Train seeds 0+ only; the eval
# battery uses --eval (held-out corpora/colors/phrasings, seeds 10000+).
subprocess.run(['python3','synth_desktop.py','--out','synth_1080','--count','200','--seed-base','0','--res','1920x1080'], check=True)
subprocess.run(['python3','synth_desktop.py','--out','synth_768','--count','120','--seed-base','2000','--res','1366x768'], check=True)

rows = []
for d in ('synth_1080','synth_768'):
    for l in open(os.path.join(d,'qa.jsonl')):
        r = json.loads(l); r['image'] = os.path.join(d, r['image']); rows.append(r)
import random as _r; _r.Random(0).shuffle(rows)
rows = rows[:len(ds)]   # 1:1 mix with the ScreenQA subset

def synth_to_chat(ex):
    return {'messages': [
        {'role':'user','content':[{'type':'image'},{'type':'text','text':ex['question']}]},
        {'role':'assistant','content':[{'type':'text','text':ex['answer']}]},
    ], 'image': _small(ex['image'])}

def _small(path, side=448):
    # 448px is the light config: the free-T4 run stalled at step 3 of 375 with
    # full-size scenes, and SmolVLM resizes to 512px max anyway, so the extra
    # pixels only cost host memory and decode time.
    im = Image.open(path).convert('RGB')
    im.thumbnail((side, side))
    return im

synth_ds = datasets.Dataset.from_list(rows)
synth_ds = synth_ds.map(synth_to_chat, remove_columns=[c for c in synth_ds.column_names if c not in ('messages','image')])
# Cast both sides to the same Image feature BEFORE concatenating; a mismatched
# feature type is what breaks concatenate_datasets on Colab.
ds = ds.cast_column('image', datasets.Image())
synth_ds = synth_ds.cast_column('image', datasets.Image())
ds = datasets.concatenate_datasets([ds, synth_ds]).shuffle(seed=0)
len(ds)


## Model + LoRA

Vision tower frozen, LoRA on the language backbone (r=32). 4-bit base
keeps it inside the free T4.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model
BASE = 'HuggingFaceTB/SmolVLM-256M-Instruct'
proc = AutoProcessor.from_pretrained(BASE)
# T4 (free Colab) has no bf16 support: load fp16, not 4-bit/bf16.
model = AutoModelForImageTextToText.from_pretrained(BASE, dtype=torch.float16, device_map='auto')
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], task_type=None))
model.print_trainable_parameters()


## Train


In [ ]:
from trl import SFTConfig, SFTTrainer
def collate(batch):
    texts = [proc.apply_chat_template(b['messages'], add_generation_prompt=False) for b in batch]
    images = [[b['image']] for b in batch]
    out = proc(text=texts, images=images, return_tensors='pt', padding=True)
    out['labels'] = out['input_ids'].clone()
    return out
cfg = SFTConfig(output_dir='./smolvlm-screen-lora', per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=1e-4, logging_steps=10, save_steps=100,
    save_total_limit=2, fp16=True, dataloader_num_workers=2,
    gradient_checkpointing=True, max_length=2048, dataset_num_proc=1, report_to=[])
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, data_collator=collate)
# fp16 base + fp16 adapter params crash the GradScaler ('Attempting to
# unscale FP16 gradients'). Keep the trainable adapter in fp32; the frozen
# base stays fp16, so the memory budget is unchanged.
for _p in trainer.model.parameters():
    if _p.requires_grad:
        _p.data = _p.data.float()
trainer.train()
trainer.save_model()


## Merge + export

After this notebook: convert to GGUF with llama.cpp's converter (CPU,
runs anywhere), then run the FIXED eval battery
(experiments/screen-fork/eval_battery.py, method=fork) against stock
256M/500M and the OCR hybrid on the same held-out scenes (seeds 10000+).
Step-by-step: experiments/screen-fork/RUN.md. Ship only if it wins.



In [ ]:
merged = model.merge_and_unload()  # needs enough RAM; on Colab this is the tight step - restart and merge on CPU if it OOMs
merged.save_pretrained('./smolvlm-screen-merged')
proc.save_pretrained('./smolvlm-screen-merged')
print('then: python convert_hf_to_gguf.py ./smolvlm-screen-merged (from llama.cpp)')
